# FFT and IFFT: Signal Recovery Demo

This notebook demonstrates the relationship between the Fast Fourier Transform (FFT) and the Inverse Fast Fourier Transform (IFFT).

**The Process:**
1.  **Create a Signal:** We'll start by creating a signal in the **time domain**. To make it interesting, we'll combine several sine waves of different frequencies and amplitudes.
2.  **Apply FFT:** We'll use `np.fft.fft` to transform the signal into the **frequency domain**. This will allow us to see the individual frequencies that make up our complex signal.
3.  **Apply IFFT:** We'll then take the frequency-domain data and use `np.fft.ifft` to transform it back into the **time domain**.

The key takeaway is that the IFFT perfectly reconstructs the original signal, demonstrating that the Fourier Transform is a reversible process. No information is lost in the transformation.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Set up the plotting style
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Create a Composite Signal

Let's create a signal by adding three different sine waves together. This will simulate a more realistic signal that isn't just a single tone.

In [ ]:
# --- Signal Parameters ---
sampling_rate = 1000  # Hz
duration = 2          # seconds

# Create the time vector
N = int(sampling_rate * duration)
t = np.linspace(0, duration, N, endpoint=False)

# --- Components of our signal ---
# Frequency (Hz), Amplitude
components = {
    20: 2.0,  # 20 Hz with amplitude 2.0
    50: 1.5,  # 50 Hz with amplitude 1.5
    120: 0.8  # 120 Hz with amplitude 0.8
}

# Create the composite signal by adding the sine waves
original_signal = np.zeros(N)
for freq, amp in components.items():
    original_signal += amp * np.sin(2 * np.pi * freq * t)

# --- Plot the Original Signal (Time Domain) ---
plt.figure(figsize=(14, 6))
plt.plot(t, original_signal, label='Original Signal')
plt.title('Original Composite Signal (Time Domain)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.xlim(0, 0.2) # Zoom in to see the waveform shape
plt.legend()
plt.grid(True)
plt.show()

The resulting signal has a complex waveform because it's a mix of multiple frequencies.

## 2. Apply FFT to Analyze the Frequencies

Now, we'll apply the FFT to see if we can identify the original frequencies (20, 50, and 120 Hz) that we put into the signal.

In [ ]:
# --- Perform FFT ---
# The FFT result is an array of complex numbers
fft_result = np.fft.fft(original_signal)

# --- Calculate corresponding frequencies ---
freq_bins = np.fft.fftfreq(N, 1 / sampling_rate)

# We only need the positive frequencies for plotting
half_N = N // 2
positive_freqs = freq_bins[:half_N]
fft_magnitude = np.abs(fft_result[:half_N]) * (2 / N) # Normalize amplitude

# --- Plot the Frequency Spectrum ---
plt.figure(figsize=(14, 6))
plt.plot(positive_freqs, fft_magnitude)
plt.title('Frequency Spectrum (FFT)')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Amplitude')
plt.grid(True)
# Find peaks to verify
for freq, amp in components.items():
    plt.axvline(x=freq, color='r', linestyle='--', label=f'{freq} Hz Peak' if freq==20 else "")
plt.legend()
plt.show()

Success! The FFT plot clearly shows three peaks at 20 Hz, 50 Hz, and 120 Hz, with amplitudes matching the ones we defined (2.0, 1.5, and 0.8). This confirms the FFT correctly decomposed our signal.

## 3. Apply IFFT to Recover the Original Signal

Now for the main event. We will take the `fft_result` (the complex numbers from the frequency domain) and feed it into the IFFT to see if we get our original time-domain signal back.

In [ ]:
# --- Perform Inverse FFT ---
# We use the full (unmodified) fft_result as input
recovered_signal = np.fft.ifft(fft_result)

# The result of ifft is complex, but since our original signal was real,
# the imaginary part will be negligible (due to floating point errors). 
# We take the real part.
recovered_signal = np.real(recovered_signal)

# --- Plot and Compare ---
plt.figure(figsize=(14, 6))
plt.plot(t, original_signal, label='Original Signal', color='blue', linewidth=4)
plt.plot(t, recovered_signal, label='Recovered Signal (from IFFT)', color='red', linestyle='--', linewidth=2)
plt.title('Original Signal vs. Recovered Signal')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.xlim(0, 0.2)
plt.legend()
plt.grid(True)
plt.show()

The plot shows the red dashed line (Recovered Signal) perfectly overlapping the solid blue line (Original Signal). They appear as one line, providing strong visual proof of recovery.

### Verification with an Error Calculation

To be absolutely sure, we can calculate the difference between the original and recovered signals. Due to tiny floating-point computer math errors, it won't be exactly zero, but it should be extremely close.

In [ ]:
# Calculate the Mean Squared Error (MSE)
mse = np.mean((original_signal - recovered_signal)**2)

print(f"The Mean Squared Error between the original and recovered signal is: {mse}")
print("A very small error (e.g., e-30) confirms the signals are virtually identical.")

## Conclusion

This demonstration shows that the **FFT** and **IFFT** are inverse operations. 

1. We can take any time-domain signal and use **FFT** to view its constituent frequencies.
2. We can then take that frequency-domain representation and use **IFFT** to perfectly reconstruct the original time-domain signal.

This round-trip capability is fundamental to digital signal processing, enabling filtering, compression, and many other advanced techniques.